# ComplianceGPT — Option A Contract Acceptance Test (Clean Run)
This notebook validates **Generator → Pipeline → Verifier** behavior under **Option A**:
- `PRESERVE` == `PARAMS_REQUIRED` + **literal placeholder preservation**
- `NO_EVIDENCE` == **no injected spans**
- Gold-row runs must return `verifier_pass=True` (no hard errors)


## 0) Configuration (EDIT ME)
Set the **Drive root** and file paths once here.
All other cells use these values.

## 1) Mount Drive (Colab)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
DRIVE_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"

CCS_REV5 = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
CCS_REV4 = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

# Gold sets (patched with PRESERVE rows)
GOLD_REV5 = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev5_gold-set_100q.csv"
GOLD_REV4 = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev4_gold-set_36q.csv"

ANSWERER_PIPELINE_DIR = f"{DRIVE_ROOT}/src/compliancegpt/pipeline"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# Runtime knobs
TOP_K = 10
DOC_FILTER_MODE = "all"        # recommended for clause-level CCS
USE_GENERATOR = True           # True = end-to-end; False = pipeline-only extractive fallback
STRICT_CCS_ASSERT = True
MIN_CCS_DOCS = 1000


## 2) Imports + Path Setup
add answerer folder to `sys.path` and imports the pipeline class.

In [3]:
import os, sys
from pathlib import Path

# IMPORTANT: add the repo's /src (package root), not a subfolder.
SRC_DIR = str(Path(DRIVE_ROOT, "src").resolve())
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
print("[sys.path] added:", SRC_DIR)

# Sanity check files exist
for fp in [CCS_REV5, CCS_REV4, GOLD_REV5, GOLD_REV4]:
    print("exists?", fp, "->", Path(fp).exists())

from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline


[sys.path] added: /content/drive/MyDrive/ComplianceGPT_v2/src
exists? /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl -> True
exists? /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl -> True
exists? /content/drive/MyDrive/ComplianceGPT_v2/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev5_gold-set_100q.csv -> True
exists? /content/drive/MyDrive/ComplianceGPT_v2/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev4_gold-set_36q.csv -> True
[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib


## 3) Load Gold Sets + Quick Sanity
We verify `PRESERVE` exists and print counts.

In [4]:
import pandas as pd

def read_csv_robust(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"[OK] read with encoding={enc}, rows={len(df)}, cols={len(df.columns)}")
            return df
        except UnicodeDecodeError as e:
            print(f"[FAIL] encoding={enc}: {e}")
    # last resort: force-read and replace bad characters
    df = pd.read_csv(path, encoding="utf-8", encoding_errors="replace")
    print(f"[WARN] forced read with replacement, rows={len(df)}")
    return df

gold5 = read_csv_robust(GOLD_REV5)
gold4 = read_csv_robust(GOLD_REV4)


[FAIL] encoding=utf-8: 'utf-8' codec can't decode byte 0xd1 in position 529: invalid continuation byte
[FAIL] encoding=utf-8-sig: 'utf-8' codec can't decode byte 0xd1 in position 47789: invalid continuation byte
[OK] read with encoding=cp1252, rows=100, cols=12
[OK] read with encoding=utf-8, rows=36, cols=12


## 4) Initialize Pipelines (Rev5 + Rev4)
This loads the model once per pipeline and initializes retriever/generator/verifier.

In [5]:
!pip -q install -U "bitsandbytes>=0.46.1" sentence-transformers faiss-cpu rank-bm25 accelerate

import torch
if torch.cuda.is_available():
    !pip -q install bitsandbytes
print("cuda:", torch.cuda.is_available(), "torch:", torch.__version__)


cuda: True torch: 2.9.0+cu126


In [6]:

pipe5 = ComplianceGPTPipeline(
    framework_version="rev5",
    model_id=MODEL_ID,
    use_qur=True,
    doc_filter_mode=DOC_FILTER_MODE,
    ccs_path=CCS_REV5,
    strict_ccs_assert=STRICT_CCS_ASSERT,
    min_ccs_docs=MIN_CCS_DOCS,
)

pipe4 = ComplianceGPTPipeline(
    framework_version="rev4",
    model_id=MODEL_ID,
    use_qur=True,
    doc_filter_mode=DOC_FILTER_MODE,
    ccs_path=CCS_REV4,
    strict_ccs_assert=STRICT_CCS_ASSERT,
    min_ccs_docs=MIN_CCS_DOCS,
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`torch_dtype` is deprecated! Use `dtype` instead!


...

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 5) Acceptance Test A — Rev5 PRESERVE (id=1)
Expected:
- `status=PARAMS_REQUIRED`
- placeholders preserved
- `verifier_pass=True`

In [7]:

row = gold5[gold5["id"] == 1].iloc[0].to_dict()
out = pipe5.answer(row["question"], gold_row=row, top_k=TOP_K, use_generator=USE_GENERATOR)

c = out["contract"]
print("status:", c.get("status"))
print("odp_required_list:", c.get("odp_required_list"))
print("evidence_spans_len:", len(c.get("evidence_spans", [])))
print("answer_text_preview:", repr((c.get("answer_text","") or "")[:120]))
print("verifier_pass:", c.get("verifier_pass"))
print("verifier_errors:", c.get("verifier_errors", []))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


status: ERROR
odp_required_list: []
evidence_spans_len: 0
answer_text_preview: ''
verifier_pass: False
verifier_errors: []


## 6) Acceptance Test B — Rev4 PRESERVE (id=21)
Expected:
- `status=PARAMS_REQUIRED`
- placeholders preserved
- `verifier_pass=True`

In [8]:

row = gold4[gold4["id"] == 21].iloc[0].to_dict()
out = pipe4.answer(row["question"], gold_row=row, top_k=TOP_K, use_generator=USE_GENERATOR)

c = out["contract"]
print("status:", c.get("status"))
print("odp_required_list:", c.get("odp_required_list"))
print("evidence_spans_len:", len(c.get("evidence_spans", [])))
print("answer_text_preview:", repr((c.get("answer_text","") or "")[:120]))
print("verifier_pass:", c.get("verifier_pass"))
print("verifier_errors:", c.get("verifier_errors", []))


status: PARAMS_REQUIRED
odp_required_list: ['pe-16_prm_1', 'pe-3_prm_1', 'pe-3_prm_2', 'pe-3_prm_4', 'pe-3_prm_5', 'pe-3_prm_6', 'pe-3_prm_7', 'pe-3_prm_8', 'pe-3_prm_9']
evidence_spans_len: 2
answer_text_preview: 'The organization: a. Enforces physical access authorizations at {{ insert: param, pe-3_prm_1 }} by; 1. Verifying individ'
verifier_pass: False
verifier_errors: ["ODPRequiredListNotEqualGold:gold=['pe-3_prm.1', 'pe-3_prm.2', 'pe-3_prm.4', 'pe-3_prm.5', 'pe-3_prm.6', 'pe-3_prm.7', 'pe-3_prm.8', 'pe-3_prm.9'],gen=['pe-16_prm.1', 'pe-3_prm.1', 'pe-3_prm.2', 'pe-3_prm.4', 'pe-3_prm.5', 'pe-3_prm.6', 'pe-3_prm.7', 'pe-3_prm.8', 'pe-3_prm.9']", "MissedControlCitations:['PE-3']", "MissedDocIds:['pe-3_smt.a', 'pe-3_smt.a.1', 'pe-3_smt.a.2', 'pe-3_smt.b', 'pe-3_smt.c', 'pe-3_smt.d', 'pe-3_smt.e', 'pe-3_smt.f', 'pe-3_smt.g']", 'LowControlRecall']


## 7) Acceptance Test C — NO_EVIDENCE Sanity
Expected:
- `status=NO_EVIDENCE`
- `evidence_spans_len=0`
- `answer_text` is `''` or `'NO_EVIDENCE'`

In [9]:

out = pipe5.answer("zzzxqv not a real control question", gold_row=None, top_k=TOP_K, use_generator=USE_GENERATOR)
c = out["contract"]

print("status:", c.get("status"))
print("selected_source_ids:", c.get("selected_source_ids", []))
print("odp_required_list:", c.get("odp_required_list", []))
print("evidence_spans_len:", len(c.get("evidence_spans", [])))
print("answer_text_preview:", repr((c.get("answer_text","") or "")[:120]))


status: NO_EVIDENCE
selected_source_ids: []
odp_required_list: []
evidence_spans_len: 0
answer_text_preview: ''


## 8) Small Batch Sanity Run (10 rows each)
This is not the full evaluation; it’s a quick regression check.
We report pass rates overall and by `resolution_policy`.

In [10]:

import random
import pandas as pd

def run_batch(pipe, df, n=10, seed=7):
    rng = random.Random(seed)
    idxs = list(df.index)
    rng.shuffle(idxs)
    idxs = idxs[:min(n, len(idxs))]

    rows = []
    for i in idxs:
        row = df.loc[i].to_dict()
        out = pipe.answer(row["question"], gold_row=row, top_k=TOP_K, use_generator=USE_GENERATOR)
        c = out["contract"]
        rows.append({
            "id": row.get("id"),
            "control_id": row.get("control_id"),
            "resolution_policy": str(row.get("resolution_policy","")).upper().strip(),
            "status": str(c.get("status","")).upper().strip(),
            "verifier_pass": bool(c.get("verifier_pass", False)),
            "n_spans": len(c.get("evidence_spans", []) or []),
            "errors": ";".join((c.get("verifier_errors", []) or [])[:5]),
        })
    return pd.DataFrame(rows)

df5 = run_batch(pipe5, gold5, n=10, seed=5)
df4 = run_batch(pipe4, gold4, n=10, seed=4)

print("=== rev5 sample ===")
display(df5)
print("\n=== rev4 sample ===")
display(df4)

def summarize(df):
    overall = float(df["verifier_pass"].mean()) if len(df) else 0.0
    by = df.groupby("resolution_policy")["verifier_pass"].mean().sort_index() if len(df) else None
    return overall, by

overall5, by5 = summarize(df5)
overall4, by4 = summarize(df4)

print("\nrev5 overall pass_rate:", overall5)
print(by5)
print("\nrev4 overall pass_rate:", overall4)
print(by4)

fails5 = df5[df5["verifier_pass"] == False]
fails4 = df4[df4["verifier_pass"] == False]
if len(fails5):
    print("\nrev5 failures:")
    display(fails5)
if len(fails4):
    print("\nrev4 failures:")
    display(fails4)


=== rev5 sample ===


,id,control_id,resolution_policy,status,verifier_pass,n_spans,errors
0,92,SI-3,ASK,PARAMS_REQUIRED,False,3,ODPRequiredListNotEqualGold:gold=['si-3_odp.1'...
1,3,AC-6,NAN,OK,False,2,MissedControlCitations:['AC-6'];LowControlRecall
2,31,IA-2,NAN,OK,False,2,MissedControlCitations:['IA-2'];LowControlRecall
3,34,IA-6,NAN,OK,False,2,MissedControlCitations:['IA-6'];LowControlRecall
4,59,PL-8,FILL_FROM_PROFILE,OK,False,2,MissingODPsButStatusNotParamsRequired:['pl-8_o...
5,69,PS-5,PRESERVE,PARAMS_REQUIRED,False,9,PreserveMissingPlaceholder:ps-05_odp.01;Preser...
6,86,SC-7,FILL_FROM_PROFILE,PARAMS_REQUIRED,False,2,"ODPRequiredListNotEqualGold:gold=['sc-7_odp'],..."
7,8,AT-4,FILL_FROM_PROFILE,PARAMS_REQUIRED,False,2,"ODPRequiredListNotEqualGold:gold=['at-4_odp'],..."
8,75,PT-6,NAN,OK,False,3,MissedControlCitations:['PT-6'];MissedDocIds:[...
9,30,CP-9,PRESERVE,PARAMS_REQUIRED,False,6,ODPRequiredListNotEqualGold:gold=['cp-09_odp.0...



=== rev4 sample ===


,id,control_id,resolution_policy,status,verifier_pass,n_spans,errors
0,25,PM-1,NAN,PARAMS_REQUIRED,False,3,MissedControlCitations:['PM-1'];LowControlRecall
1,24,PL-4,NAN,PARAMS_REQUIRED,False,2,MissedControlCitations:['PL-4'];LowControlRecall
2,23,PL-2,NAN,OK,False,2,MissedControlCitations:['PL-2'];MissedDocIds:[...
3,14,IA-5,NAN,PARAMS_REQUIRED,False,2,MissedControlCitations:['IA-5'];LowControlRecall
4,34,SC-8,FILL_FROM_PROFILE,OK,False,5,MissingODPsButStatusNotParamsRequired:['sc-8_p...
5,21,PE-3,PRESERVE,PARAMS_REQUIRED,False,2,ODPRequiredListNotEqualGold:gold=['pe-3_prm.1'...
6,36,SI-4,ASK,PARAMS_REQUIRED,False,3,ODPRequiredListNotEqualGold:gold=['si-4_prm.1'...
7,27,PS-2,NAN,PARAMS_REQUIRED,False,2,MissedControlCitations:['PS-2'];MissedDocIds:[...
8,29,RA-2,NAN,OK,False,3,MissedControlCitations:['RA-2'];LowControlRecall
9,15,IR-4,NAN,OK,False,2,MissedControlCitations:['IR-4'];LowControlRecall



rev5 overall pass_rate: 0.0
resolution_policy
ASK                  0.0
FILL_FROM_PROFILE    0.0
NAN                  0.0
PRESERVE             0.0
Name: verifier_pass, dtype: float64

rev4 overall pass_rate: 0.0
resolution_policy
ASK                  0.0
FILL_FROM_PROFILE    0.0
NAN                  0.0
PRESERVE             0.0
Name: verifier_pass, dtype: float64

rev5 failures:


,id,control_id,resolution_policy,status,verifier_pass,n_spans,errors
0,92,SI-3,ASK,PARAMS_REQUIRED,False,3,ODPRequiredListNotEqualGold:gold=['si-3_odp.1'...
1,3,AC-6,NAN,OK,False,2,MissedControlCitations:['AC-6'];LowControlRecall
2,31,IA-2,NAN,OK,False,2,MissedControlCitations:['IA-2'];LowControlRecall
3,34,IA-6,NAN,OK,False,2,MissedControlCitations:['IA-6'];LowControlRecall
4,59,PL-8,FILL_FROM_PROFILE,OK,False,2,MissingODPsButStatusNotParamsRequired:['pl-8_o...
5,69,PS-5,PRESERVE,PARAMS_REQUIRED,False,9,PreserveMissingPlaceholder:ps-05_odp.01;Preser...
6,86,SC-7,FILL_FROM_PROFILE,PARAMS_REQUIRED,False,2,"ODPRequiredListNotEqualGold:gold=['sc-7_odp'],..."
7,8,AT-4,FILL_FROM_PROFILE,PARAMS_REQUIRED,False,2,"ODPRequiredListNotEqualGold:gold=['at-4_odp'],..."
8,75,PT-6,NAN,OK,False,3,MissedControlCitations:['PT-6'];MissedDocIds:[...
9,30,CP-9,PRESERVE,PARAMS_REQUIRED,False,6,ODPRequiredListNotEqualGold:gold=['cp-09_odp.0...



rev4 failures:


,id,control_id,resolution_policy,status,verifier_pass,n_spans,errors
0,25,PM-1,NAN,PARAMS_REQUIRED,False,3,MissedControlCitations:['PM-1'];LowControlRecall
1,24,PL-4,NAN,PARAMS_REQUIRED,False,2,MissedControlCitations:['PL-4'];LowControlRecall
2,23,PL-2,NAN,OK,False,2,MissedControlCitations:['PL-2'];MissedDocIds:[...
3,14,IA-5,NAN,PARAMS_REQUIRED,False,2,MissedControlCitations:['IA-5'];LowControlRecall
4,34,SC-8,FILL_FROM_PROFILE,OK,False,5,MissingODPsButStatusNotParamsRequired:['sc-8_p...
5,21,PE-3,PRESERVE,PARAMS_REQUIRED,False,2,ODPRequiredListNotEqualGold:gold=['pe-3_prm.1'...
6,36,SI-4,ASK,PARAMS_REQUIRED,False,3,ODPRequiredListNotEqualGold:gold=['si-4_prm.1'...
7,27,PS-2,NAN,PARAMS_REQUIRED,False,2,MissedControlCitations:['PS-2'];MissedDocIds:[...
8,29,RA-2,NAN,OK,False,3,MissedControlCitations:['RA-2'];LowControlRecall
9,15,IR-4,NAN,OK,False,2,MissedControlCitations:['IR-4'];LowControlRecall
